In [1]:
!pip install datasets
!pip install transformers

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# Use a pipeline as a high-level helper
from transformers import pipeline

# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("NepBERTa/NepBERTa")
model = AutoModelForSequenceClassification.from_pretrained("NepBERTa/NepBERTa",from_tf=True).to(device)

classifier = pipeline("sentiment-analysis", model=model, tokenizer = tokenizer)

# model_dir = "/content/drive/My Drive/NepBERTa/model/"
# model = TFAutoModelForSequenceClassification.from_pretrained(model_dir)
# tokenizer = BertTokenizer.from_pretrained(model_dir)

cuda


All TF 2.0 model weights were used when initializing BertForSequenceClassification.

All the weights of BertForSequenceClassification were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertForSequenceClassification for predictions without further training.
Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [4]:
import pandas as pd

df_train = pd.read_csv("/content/drive/My Drive/NepBERTa/train.csv")
df_test = pd.read_csv("/content/drive/My Drive/NepBERTa/test.csv")
main_path = "/content/drive/My Drive/NepBERTa/"

In [5]:
df_train

,text,label
0,बजार ले जसरी ट्रेन्ड चेन्ज गर्यो यो हेर्दा तत्...,2
1,1000 अंकले घटेको नेप्से 200 अंकले बढ्नु ठूलो क...,1
2,होइन यो सानिमा बैंक ले bonus घोसणा गरेको २ महि...,2
3,"खैँ MBJC प्रति कित्तामा रू,10/-ले बढेर आज रू,1...",2
4,राम्रो भयो️️,1
...,...,...
5995,समाज परिवर्तन गराउन लाई अरु को मुख हेर्ने भन्द...,1
5996,"Filmy क्षेत्रमा धेरै गर्नु भयो,,अब समाज र देश ...",2
5997,यस्तै यस्तै कार्यक्रम अझ बढी हुन जरुरी छ कुना ...,2
5998,बधाई र सुभकामना ।।,1


In [6]:
df_test

,text,label
0,असाध्यै राम्रो कार्यक्रम आयोजना गरिएको छ हजुरह...,1
1,"राम्रो कार्यक्रम, पहिलो सिजनले समेटेको कार्यक्...",1
2,महानायक राजेश हमाल तपाई साँच्चै धन्यवादको पात्...,1
3,जातको प्रष्न बाट सबै जनालाई सकरात्मक सन्देश मि...,1
4,"बहसको सुरुवात भएको छ, अझै जोडदार रुपमा गर्नुपर...",1
...,...,...
1991,कस्ता कस्ता पागल memory king छन यार नेपालमा,0
1992,दोस्रो मूर्ख बिजय साही हो,0
1993,बिजय शाहीलाई किन निरुत्साहित गरेको त पुण्य गौत...,0
1994,यस्ता बिदेशी महादलालीहरु कहाँ गएर यो प्रश्नको ...,0


In [7]:
df_train.dtypes, df_test.dtypes

(text     object
 label    object
 dtype: object,
 text     object
 label    object
 dtype: object)

In [8]:
df_train['label'].value_counts()

1     2378
0     2377
2     1236
-        5
20       1
11       1
o        1
--       1
Name: label, dtype: int64

In [9]:
df_test['label'].value_counts()

1    888
0    610
2    496
o      1
-      1
Name: label, dtype: int64

In [10]:
# For Train
df_train.drop(df_train[df_train['label'] == '-'].index, inplace = True)
df_train.drop(df_train[df_train['label'] == '20'].index, inplace = True)
df_train.drop(df_train[df_train['label'] == '11'].index, inplace = True)
df_train.drop(df_train[df_train['label'] == 'o'].index, inplace = True)
df_train.drop(df_train[df_train['label'] == '--'].index, inplace = True)

# For Test
df_test.drop(df_test[df_test['label'] == '-'].index, inplace = True)
df_test.drop(df_test[df_test['label'] == 'o'].index, inplace = True)

In [11]:
df_train['label'].value_counts()

1    2378
0    2377
2    1236
Name: label, dtype: int64

In [12]:
df_test['label'].value_counts()

1    888
0    610
2    496
Name: label, dtype: int64

In [13]:
label_mapping = {'0':0, '1':1, '2':2}

df_train['label'] = df_train['label'].map(label_mapping)
df_test['label'] = df_test['label'].map(label_mapping)

df_train.dtypes, df_test.dtypes

(text     object
 label     int64
 dtype: object,
 text     object
 label     int64
 dtype: object)

In [14]:
df_train.isna().sum(), df_test.isna().sum()

(text     1
 label    0
 dtype: int64,
 text     1
 label    0
 dtype: int64)

In [15]:
df_train = df_train.dropna()
df_test = df_test.dropna()
df_train.isna().sum(), df_test.isna().sum()

(text     0
 label    0
 dtype: int64,
 text     0
 label    0
 dtype: int64)

In [16]:
df_train.head()

,text,label
0,बजार ले जसरी ट्रेन्ड चेन्ज गर्यो यो हेर्दा तत्...,2
1,1000 अंकले घटेको नेप्से 200 अंकले बढ्नु ठूलो क...,1
2,होइन यो सानिमा बैंक ले bonus घोसणा गरेको २ महि...,2
3,"खैँ MBJC प्रति कित्तामा रू,10/-ले बढेर आज रू,1...",2
4,राम्रो भयो️️,1


In [17]:
df_test.sample(5)

,text,label
387,उच्च सम्मान छ यो उदाहरणीय जोडि लाई ....लक्ष्मि...,1
225,जातीय बिबेद हटाउन को लागी मेरो बिचार मा सरकार ...,0
1727,दाइलाई दबाइ गर्नुको लागि पो टिकटकमा गिफ्टको ला...,0
1971,Youtube le बाहिर ल्याउदै फेरि ला पत्ता बनाउदै...,2
1795,नरे पुन्य ले जे होस युटुबेको चुलो बालिदेको छ,1


In [18]:
from datasets import Dataset, DatasetDict

df_trainset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

final_dataset = DatasetDict({
    'train':df_trainset,
    'test': test_dataset,
})

In [19]:
def tokenize_texts(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128, return_tensors='pt')

In [20]:
encoded_data = final_dataset.map(tokenize_texts, batched=True, batch_size=None)

Map:   0%|          | 0/5990 [00:00<?, ? examples/s]

Map:   0%|          | 0/1993 [00:00<?, ? examples/s]

In [21]:
encoded_data

DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5990
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1993
    })
})

In [22]:
def extract_hidden_states(batch):
  inputs = {k:v.to(device) for k,v in batch.items() if k in tokenizer.model_input_names}
    # Extract last hidden states
  with torch.no_grad():
      last_hidden_state = model(**inputs).logits
          # Return vector for [CLS] token
      return {"hidden_state": last_hidden_state[:,0].cpu().numpy()}

In [23]:
encoded_data.set_format("torch",columns=['input_ids', "attention_mask", "label"])

In [24]:
text = "असाध्यै राम्रो कार्यक्रम आयोजना गरिएको छ."

# Tokenize the text
inputs = tokenizer(text, return_tensors="pt").to(device)

# Forward pass through the model
outputs = model(**inputs)

# Access the last hidden states
last_hidden_states = outputs.logits

In [25]:
encoded_data = encoded_data.map(extract_hidden_states, batched=True)

Map:   0%|          | 0/5990 [00:00<?, ? examples/s]

Map:   0%|          | 0/1993 [00:00<?, ? examples/s]

In [26]:
from transformers import AutoModelForSequenceClassification
num_labels = 3
model = (AutoModelForSequenceClassification
         .from_pretrained("NepBERTa/NepBERTa", num_labels=num_labels, from_tf=True)
         .to(device))

All TF 2.0 model weights were used when initializing BertForSequenceClassification.

All the weights of BertForSequenceClassification were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertForSequenceClassification for predictions without further training.


In [27]:
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [28]:
# !pip install transformers[torch]
# !pip install accelerate -U
# !pip install torch
%pip install "accelerate>=0.16.0,<1" "transformers[torch]>=4.28.1,<5" "torch>=1.13.1,<2"

In [29]:
from transformers import Trainer, TrainingArguments
batch_size = 64
logging_steps = len(encoded_data["train"])
model_name = main_path+"/model"
training_args = TrainingArguments(output_dir='/content/drive/MyDrive',
                                  num_train_epochs=5,
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,
                                  evaluation_strategy="epoch",
                                  disable_tqdm=False,
                                  logging_steps=logging_steps,
                                  push_to_hub=False,
                                  log_level="error")

In [30]:
trainer = Trainer(model=model, args=training_args,
                      compute_metrics=compute_metrics,train_dataset=encoded_data["train"],
                      eval_dataset=encoded_data["test"],tokenizer=tokenizer)
trainer.train();

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.797290,0.647767,0.615604
2,No log,0.817661,0.649272,0.643440
3,No log,0.842143,0.648771,0.649955
4,No log,0.864247,0.654792,0.653534
5,No log,0.880905,0.656799,0.652168


In [31]:
preds_output = trainer.predict(encoded_data["test"])

In [32]:
preds_output.metrics

{'test_loss': 0.8809053897857666,
 'test_accuracy': 0.6567987957852484,
 'test_f1': 0.6521682817481627,
 'test_runtime': 15.2,
 'test_samples_per_second': 131.119,
 'test_steps_per_second': 2.105}

In [37]:
trainer.save_model("/content/final")

In [86]:
custom_text = "म तिमिलाइ राम्रो साथि मान्छु"
pipe = pipeline("text-classification", model='/content/final')
preds = pipe(custom_text)

In [87]:
preds

[{'label': 'LABEL_1', 'score': 0.9461116194725037}]